<a href="https://colab.research.google.com/github/erica-teoh/stargazers-log/blob/main/MMA3001_CO2_Reconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MMA3001 Project — Reconstruction of Missing Indoor CO₂ Sensor Measurements

This project investigates the reconstruction of missing indoor CO₂ sensor measurements using numerical interpolation and machine-learning methods.

The primary engineering question is:

How accurately can missing CO₂ measurements be reconstructed using numerical interpolation and machine-learning methods, and how does reconstruction performance change as the duration of missing data increases?

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import glob
import pandas as pd
import numpy as np
import json

filename = "5EnvSensor_MayToDec2024_180kRows.csv"

matches = glob.glob(
    f"/content/drive/MyDrive/**/{filename}",
    recursive=True
)

print("Files found:")
print(matches)

Files found:
['/content/drive/MyDrive/MMA3001 Project/5EnvSensor_MayToDec2024_180kRows.csv']


In [3]:
file_path = matches[0]

raw_df = pd.read_csv(file_path)

print("Dataset shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())

raw_df.head()

Dataset shape: (179447, 8)

Columns:
['id', 'sensorid', 'devicetype', 'jsondata', 'status', 'processing_errors', 'createdate', 'processdate']


,id,sensorid,devicetype,jsondata,status,processing_errors,createdate,processdate
0,2,6012002000241,env,"[{""values"": [{""time"": 1702273965, ""value"": 4.1...",Processed,NaN,2024-02-29 03:11:58.210358+11,2024-02-29 03:11:58.210358+11
1,4,6012002000241,env,"[{""values"": [{""time"": 1702273965, ""value"": 4.1...",Processed,NaN,2024-02-29 17:09:08.833822+11,2024-02-29 17:09:08.833822+11
2,5,6012002000326,env,"[{""values"": [{""time"": 1702270857, ""value"": 4.1...",Processed,NaN,2024-02-29 17:09:12.444245+11,2024-02-29 17:09:12.444245+11
3,16,6012002000777,env,"[{""values"": [{""time"": 1683326565, ""value"": 4.1...",Processed,NaN,2024-02-29 17:10:24.443407+11,2024-02-29 17:10:24.443407+11
4,24,6012002000869,env,"[{""values"": [{""time"": 1709186696, ""value"": 4.1...",Processed,NaN,2024-02-29 17:11:11.348578+11,2024-02-29 17:11:11.348578+11


In [7]:
def unpack_sensor_row(json_string):
    """
    Extract environmental sensor measurements from one JSON record.
    Missing measurements are stored as NaN rather than causing an error.
    """

    data = json.loads(json_string)

    output = {}
    measurement_times = []

    for item in data:

        variable_name = item["variable"]["name"]
        values = item.get("values", [])

        if len(values) > 0:

            # Some entries contain a timestamp but no measurement value
            output[variable_name] = values[0].get("value", np.nan)

            if "time" in values[0]:
                measurement_times.append(values[0]["time"])

    if measurement_times:
        output["measurement_time"] = measurement_times[0]

    return output


parsed_measurements = pd.DataFrame(
    raw_df["jsondata"]
    .apply(unpack_sensor_row)
    .tolist()
)

parsed_measurements["sensorid"] = raw_df["sensorid"].values

parsed_measurements.head()

,Battery,Carbon dioxide,Humidity,Formaldehyde,Pressure,Temperature,Light Volatile Organic Compounds,Particulate matter 1,Particulate matter 4,Particulate matter 2.5,Particulate matter 10,measurement_time,sensorid
0,4.1875,603,66,6.0,1011,23.8,14.0,8.0,9.0,9.0,9.0,1702273965,6012002000241
1,4.1875,603,66,6.0,1011,23.8,14.0,8.0,9.0,9.0,9.0,1702273965,6012002000241
2,4.1875,525,68,5.0,1012,23.8,7.0,6.0,6.0,6.0,6.0,1702270857,6012002000326
3,4.1562,425,51,1.0,1015,18.8,71.0,1.0,3.0,3.0,4.0,1683326565,6012002000777
4,4.1250,465,56,17.0,1010,24.3,0.0,2.0,2.0,2.0,2.0,1709186696,6012002000869


In [8]:
sensor_counts = parsed_measurements["sensorid"].value_counts()

print(sensor_counts)

sensorid
6012002000326    58909
6012002000241    58908
6012002000869    29341
6012002000227    21363
6012002000777    10926
Name: count, dtype: int64


In [9]:
#isolate chosen sensor

SENSOR_ID = 6012002000869

sensor_df = parsed_measurements[
    parsed_measurements["sensorid"] == SENSOR_ID
].copy()

sensor_df["timestamp"] = pd.to_datetime(
    sensor_df["measurement_time"],
    unit="s",
    utc=True
)

sensor_df["timestamp"] = sensor_df["timestamp"].dt.tz_convert(
    "Australia/Melbourne"
)

sensor_df = sensor_df.sort_values("timestamp")

print("Raw rows for selected sensor:", len(sensor_df))
print("Start:", sensor_df["timestamp"].min())
print("End:", sensor_df["timestamp"].max())

sensor_df.head()

Raw rows for selected sensor: 29341
Start: 2024-02-29 17:04:56+11:00
End: 2024-06-14 03:45:44+10:00


,Battery,Carbon dioxide,Humidity,Formaldehyde,Pressure,Temperature,Light Volatile Organic Compounds,Particulate matter 1,Particulate matter 4,Particulate matter 2.5,Particulate matter 10,measurement_time,sensorid,timestamp
4,4.125,465,56,17.0,1010,24.3,0.0,2.0,2.0,2.0,2.0,1709186696,6012002000869,2024-02-29 17:04:56+11:00
8,4.125,445,56,17.0,1010,24.0,0.0,2.0,5.0,4.0,5.0,1709187297,6012002000869,2024-02-29 17:14:57+11:00
12,4.125,471,56,17.0,1010,23.8,0.0,1.0,2.0,2.0,2.0,1709188497,6012002000869,2024-02-29 17:34:57+11:00
16,4.125,457,56,17.0,1010,23.8,0.0,1.0,2.0,2.0,3.0,1709189097,6012002000869,2024-02-29 17:44:57+11:00
20,4.125,455,57,17.0,1010,23.8,0.0,1.0,2.0,2.0,2.0,1709189697,6012002000869,2024-02-29 17:54:57+11:00


In [10]:
#removing duplicate timestamps

clean_df = sensor_df.drop_duplicates(
    subset="measurement_time",
    keep="first"
).copy()

clean_df = clean_df.sort_values("timestamp").reset_index(drop=True)

print("Rows before duplicate removal:", len(sensor_df))
print("Rows after duplicate removal:", len(clean_df))
print("Duplicates removed:", len(sensor_df) - len(clean_df))

print("\nDate range:")
print(clean_df["timestamp"].min())
print("to")
print(clean_df["timestamp"].max())

Rows before duplicate removal: 29341
Rows after duplicate removal: 13800
Duplicates removed: 15541

Date range:
2024-02-29 17:04:56+11:00
to
2024-06-14 03:45:44+10:00


In [11]:
time_difference = clean_df["timestamp"].diff()

interval_minutes = time_difference.dt.total_seconds() / 60

print(interval_minutes.describe())

print("\nMost common sampling intervals:")
print(interval_minutes.round().value_counts().head(10))

count    13799.000000
mean        11.008102
std          3.013253
min          9.566667
25%         10.000000
50%         10.000000
75%         10.000000
max         29.983333
Name: timestamp, dtype: float64

Most common sampling intervals:
timestamp
10.0    12409
20.0     1389
30.0        1
Name: count, dtype: int64


In [12]:
print("Missing CO2 values:")
print(clean_df["Carbon dioxide"].isna().sum())

print("\nCO2 summary:")
print(clean_df["Carbon dioxide"].describe())

Missing CO2 values:
0

CO2 summary:
count    13800.000000
mean       473.985217
std         39.403080
min        403.000000
25%        444.000000
50%        463.000000
75%        494.000000
max        646.000000
Name: Carbon dioxide, dtype: float64
